# Paired C/D evidence-constrained pilot (v4)
Select T4 GPU and run all cells. Three cases per condition. Removes the character caps that cut off V3 explanations. Numerical feature/value combinations and comparison objects are constrained to supplied evidence. The existing response validator still checks selected-case consistency; prose support remains a separate review. Earlier outputs are preserved.


In [1]:
from pathlib import Path
import os, sys, json, hashlib, subprocess
from google.colab import drive
ROOT = Path('/content/drive/Othercomputers/My Laptop (1)/CP5403')
if not Path('/content/drive/MyDrive').is_dir():
    try:
        drive.mount('/content/drive')
    except ValueError as exc:
        raise RuntimeError('Drive connection failed. Select Runtime > Disconnect and delete runtime, reconnect and rerun this cell. Complete Google authorisation. No diagnosis has started.') from exc
if not (ROOT / 'Files/phm_condition_d.py').is_file():
    raise FileNotFoundError(f'Drive is connected but the updated project is missing at {ROOT}. Finish sync or locate CP5403 in the Files panel. Do not force-remount.')
os.chdir(ROOT)
EXPECTED = {'phm_condition_d.py': '65923a51de641781736160e31108e668d5fa7fbed37938b1bb79722e28e447ed', 'phm_literature_index.py': '725709141868ae494f1c515f67df5aec4451c0d137d461b3991c7ba436fb7f37', 'phm_diagnosis.py': 'a0b2af61d7d1a1c16512359f1e8029aae48380fc4b698c188f1f8caacf14b2fd', 'phm_diagnosis_qwen.py': 'f3c9938f7fd2638de842139db5f5d2716c57f4839860291279e45c4d47e049bb', 'test_phm_condition_d.py': '628c4d0284eeedbf92f8e4582bfcd1bd8d0277fdcd14352ec376c37bd4b9f187', 'phm_output_schema.py': '01913f0c804300723c841d7c689be5105749c1e3d9c0bc110d7a9b204ea048e9'}
for name, expected in EXPECTED.items():
    path = ROOT / 'Files' / name
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == expected, f'File not fully synced: {name}'
INDEX = ROOT / 'Datasets/PHM15-literature-D-approved-v1'
manifest = json.loads((INDEX / 'index_manifest.json').read_text(encoding='utf-8'))
for name, key in [('approved_chunks.jsonl','chunks_sha256'), ('embeddings.npy','embeddings_sha256'), ('model_assets/minilm.onnx','onnx_sha256')]:
    assert hashlib.sha256((INDEX / name).read_bytes()).hexdigest() == manifest[key], f'Index not fully synced: {name}'
print('Project and approved index verified:', manifest['chunk_count'], 'chunks')


Mounted at /content/drive
Project and approved index verified: 24 chunks


In [2]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Files/requirements-literature-colab.txt', 'onnxruntime==1.31.1', 'lm-format-enforcer==0.11.3'], check=True)
print('Dependencies installed.')


GPU: Tesla T4
Dependencies installed.


In [3]:
for test in ['test_phm_diagnosis.py', 'test_phm_condition_d.py']:
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'Files', '-p', test, '-q'], check=True)
subprocess.run([sys.executable, 'Files/phm_condition_d.py', '--root', str(ROOT), '--max-cases', '3', '--prepare-only'], check=True)
print('Paired evidence and source checks passed. The final cell runs Qwen.')


Paired evidence and source checks passed. The final cell runs Qwen.


In [4]:
subprocess.run([sys.executable, 'Files/phm_condition_d.py', '--root', str(ROOT), '--max-cases', '3'], check=True)
latest = json.loads((ROOT / 'Outputs/LATEST_RUN.json').read_text(encoding='utf-8'))
print('Latest report:', latest)
print('Let Google Drive finish syncing, then ask for the paired C/D pilot review.')


Latest report: {'run_id': '20261121T173612_paired_CD_validation_pilot_11a564be', 'run_directory': 'Outputs/runs/20261121T173612_paired_CD_validation_pilot_11a564be', 'status': 'completed', 'updated_utc': '2026-11-21T17:45:46.294257+00:00'}
Let Google Drive finish syncing, then ask for the paired C/D pilot review.
